In [ ]:
import pandas as pd
import geopandas as gpd
import sys
import os
import numpy as np

import openmatrix as omx

import openmatrix
openmatrix.numpy = np

In [ ]:
landuse_dir = r"C:\Users\USYS671257\OneDrive - WSP O365\42_Boston_CTPS\land_use_prep"
skim_dir = r"C:\Users\USYS671257\OneDrive - WSP O365\42_Boston_CTPS\skim_prep"

: 

## subarea land use 

In [ ]:
zone_file = gpd.read_file(os.path.join(landuse_dir, "zonal", "shp", "CTPS_TDM23_TAZ_2017g_v202303.shp"))
landuse_file = pd.read_csv(os.path.join(landuse_dir, "processed", "land_use.csv"))

: 

In [ ]:
downtown_taz_ids = zone_file[zone_file["ring"].isin([0,1])].taz_id.to_list()

: 

In [ ]:
landuse_file = landuse_file[landuse_file["TAZ"].isin(downtown_taz_ids)]

: 

In [ ]:
landuse_file.to_csv(os.path.join(landuse_dir, "processed", "land_use_subarea.csv"), index=False)

: 

## subarea skims

In [ ]:
skims = [
    "hwy_am_pm",
    "hwy_md_ev",
    "nm_daily",
    "tw_am_pm",
    "tw_md_ev",
    "ta_am_pm",
    "ta_md_ev",
]

mapping_name = "ID"

: 

In [ ]:
for skim in skims:
    input_omx = os.path.join(skim_dir, "processed", f"{skim}.omx")
    output_omx = os.path.join(skim_dir, "processed", "subarea", f"{skim}_subarea.omx")

    print(f"process {input_omx}")

    with omx.open_file(input_omx, "r") as matrix_in:
        mapping = matrix_in.mapping(mapping_name)
        zones = [z for z in downtown_taz_ids if z in mapping]
        locations = [mapping[z] for z in zones]
        
        zones = np.array(zones, dtype=np.int32)
        locations = np.array(locations)

        with omx.open_file(output_omx, "w") as matrix_out:
            matrix_out.create_mapping(mapping_name, zones)

            for matrix_name in matrix_in.list_matrices():
                data = matrix_in[matrix_name][:]
                subarea_data = data[np.ix_(locations,locations)]
                matrix_out[matrix_name] = subarea_data

                print(f"save matrix {matrix_name}")
    print(f"create {output_omx}")

: 